This codelab is largely based on [scikit-learn example code](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) .

In [ ]:
import warnings

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
%matplotlib inline

# Random Forest

The sklearn.ensemble module includes two averaging algorithms based on randomized decision trees: the RandomForest algorithm and the Extra-Trees method. Both algorithms are perturb-and-combine techniques specifically designed for trees. This means a diverse set of classifiers is created by introducing randomness in the classifier construction. The prediction of the ensemble is given as the averaged prediction of the individual classifiers.

As other classifiers, forest classifiers have to be fitted with two arrays: a sparse or dense array X of size [n_samples, n_features] holding the training samples, and an array Y of size [n_samples] holding the target values (class labels) for the training samples:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = [[0, 0], [1, 1]]
Y = [0, 1]
model = RandomForestClassifier(n_estimators=10)
model = model.fit(X, Y)
print(model)

In random forests (see RandomForestClassifier and RandomForestRegressor classes), each tree in the ensemble is built from a sample drawn with replacement (i.e., a bootstrap sample) from the training set. In addition, when splitting a node during the construction of the tree, the split that is chosen is no longer the best split among all features. Instead, the split that is picked is the best split among a random subset of the features. As a result of this randomness, the bias of the forest usually slightly increases (with respect to the bias of a single non-random tree) but, due to averaging, its variance also decreases, usually more than compensating for the increase in bias, hence yielding an overall better model.

In contrast to the original publication, the scikit-learn implementation combines classifiers by averaging their probabilistic prediction, instead of letting each classifier vote for a single class.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

X, y = make_classification(
  n_samples=1000,
  n_features=4,
  n_informative=2,
  n_redundant=0,
  random_state=0,
  shuffle=False,
)
model = RandomForestClassifier(max_depth=2, random_state=0)
model.fit(X, y)

print(model.feature_importances_)
print(model.predict([[0, 0, 0, 0]]))

## OOB Errors for Random Forests

The RandomForestClassifier is trained using bootstrap aggregation, where each new tree is fit from a bootstrap sample of the training observations z_i = (x_i, y_i). The out-of-bag (OOB) error is the average error for each z_i calculated using predictions from the trees that do not contain z_i in their respective bootstrap sample. This allows the RandomForestClassifier to be fit and validated whilst being trained.

The example below demonstrates how the OOB error can be measured at the addition of each new tree during training. The resulting plot allows a practitioner to approximate a suitable value of n_estimators at which the error stabilizes.

In [ ]:
from collections import OrderedDict

import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

# Author: Kian Ho <hui.kian.ho@gmail.com>
#         Gilles Louppe <g.louppe@gmail.com>
#         Andreas Mueller <amueller@ais.uni-bonn.de>
#
# License: BSD 3 Clause

print(__doc__)

RANDOM_STATE = 123

# Generate a binary classification dataset.
X, y = make_classification(
  n_samples=500,
  n_features=25,
  n_clusters_per_class=1,
  n_informative=15,
  random_state=RANDOM_STATE,
)

# NOTE: Setting the `warm_start` construction parameter to `True` disables
# support for parallelized ensembles but is necessary for tracking the OOB
# error trajectory during training.
ensemble_models = [
  (
    "RandomForestClassifier, max_features='sqrt'",
    RandomForestClassifier(
      warm_start=True,
      oob_score=True,
      max_features="sqrt",
      random_state=RANDOM_STATE,
    ),
  ),
  (
    "RandomForestClassifier, max_features='log2'",
    RandomForestClassifier(
      warm_start=True,
      max_features="log2",
      oob_score=True,
      random_state=RANDOM_STATE,
    ),
  ),
  (
    "RandomForestClassifier, max_features=None",
    RandomForestClassifier(
      warm_start=True,
      max_features=None,
      oob_score=True,
      random_state=RANDOM_STATE,
    ),
  ),
]

# Map a classifier name to a list of (<n_estimators>, <error rate>) pairs.
error_rate = OrderedDict((label, []) for label, _ in ensemble_models)

# Range of `n_estimators` values to explore.
min_estimators = 15
max_estimators = 175

for label, model in ensemble_models:
  for i in range(min_estimators, max_estimators + 1):
    model.set_params(n_estimators=i)
    model.fit(X, y)

    # Record the OOB error for each `n_estimators=i` setting.
    oob_error = 1 - model.oob_score_
    error_rate[label].append((i, oob_error))

# Generate the "OOB error rate" vs. "n_estimators" plot.
for label, model_err in error_rate.items():
  xs, ys = zip(*model_err)
  plt.plot(xs, ys, label=label)

plt.xlim(min_estimators, max_estimators)
plt.xlabel("n_estimators")
plt.ylabel("OOB error rate")
plt.legend(loc="upper right")
plt.show()

## Affichage du premier arbre de décision grâce à l'outil GraphViz de sklearn

In [ ]:
from sklearn.tree import export_graphviz

# Export as dot file
export_graphviz(
  model.estimators_[0],
  out_file="tree.dot",
  rounded=True,
  proportion=False,
  precision=2,
  filled=True,
)

# Convert to png using system command (requires Graphviz)
from subprocess import call

call(["dot", "-Tpng", "tree.dot", "-o", "tree.png", "-Gdpi=600"])
# Convert to svg
call(["dot", "-Tsvg", "tree.dot", "-o", "tree.svg"])

# Display in jupyter notebook
from IPython.display import Image

Image(filename="tree.png")

In [ ]:
from IPython.display import SVG

SVG(filename="tree.svg")